# PyTerrier sparse retrieval (BM25 / SPLADE + optional monoT5 rerank)

Notebook version of `pt_retrieval.py`. It reuses that module's `RetrievalConfig`
and `PyTerrierRetrievalPipeline` directly, so behaviour is identical to the
command-line script -- this notebook is just a more interactive way to run it,
inspect the index/config, and look at results per query.

**Environment note:** PyTerrier embeds a JVM inside the Python process via
`pyjnius`. On this machine, the system/Homebrew Python crashes trying to start
that JVM (a JIT/code-signing restriction), while the Anaconda Python
distribution (`/opt/anaconda3/bin/python3`) does not. If you hit a JVM crash
running this notebook, pick the Anaconda Python as this notebook's kernel.

## 1. Setup

In [1]:
import sys, os

# So `import pt_retrieval` finds the module next to this notebook regardless
# of the working directory the kernel was started from.
PROJECT_DIR = os.path.abspath(".")
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

import pandas as pd
from pt_retrieval import RetrievalConfig, PyTerrierRetrievalPipeline, is_danish_collection

pd.set_option("display.max_colwidth", 80)

## 2. Configure the run

Edit these to point at a different `*_chunks_docs.csv` / `*_queries_topic_names_qid_query.csv`
pair, switch retriever, or turn on monoT5 reranking.

In [2]:
DOCS_CSV = "euaa_asylum_report_chunks_docs.csv"
TOPICS_CSV = "euaa_asylum_report_queries_topic_names_qid_query.csv"
RETRIEVER = "bm25"          # "bm25" | "splade"
TOP_K = 100
LANGUAGE = "auto"           # "auto" | "danish" | "other"
RERANK_MONOT5 = False

print(f"Danish collection detected: {is_danish_collection(DOCS_CSV, LANGUAGE)}")

config = RetrievalConfig(
    docs_csv=DOCS_CSV,
    topics_csv=TOPICS_CSV,
    retriever=RETRIEVER,
    top_k=TOP_K,
    language=LANGUAGE,
    rerank_monot5=RERANK_MONOT5,
)
config

Danish collection detected: False


RetrievalConfig(docs_csv='euaa_asylum_report_chunks_docs.csv', topics_csv='euaa_asylum_report_queries_topic_names_qid_query.csv', retriever='bm25', top_k=100, language='auto', index_dir=None, overwrite_index=False, rerank_monot5=False, splade_model='naver/splade-cocondenser-ensembledistil', output_csv=None)

## 3. Run the pipeline

`run()` builds the Terrier index (or reuses one already built at the same
`index_dir`), retrieves the top-k per query, joins in the retrieved
document's text, optionally reranks with monoT5, and writes the output CSV --
all in one call. It also returns the resulting DataFrame.

In [3]:
pipeline = PyTerrierRetrievalPipeline(config)
results_df = pipeline.run()

output_path = config.output_csv or pipeline._default_output_path()
print(f"Retrieved {len(results_df)} rows ({results_df['qid'].nunique()} queries) -> {output_path}")
results_df.head(10)

Java started and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
/Users/qbr926/Desktop/actor/pt_retrieval.py:114: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()



Retrieved 285 rows (5 queries) -> euaa_asylum_report_queries_topic_names_qid_query_bm25.csv


,qid,docid,docno,rank,score,query,text
0,0,403,151_chunk001,0,5.851850,Asylum Definition And Assessment,None of them possessed documentary evidence supporting their claimed ages an...
1,0,423,201_chunk001,1,5.638718,Asylum Definition And Assessment,It was submitted on behalf of the applicant that a family could be a social ...
2,0,1,5442_chunk001,2,5.495891,Asylum Definition And Assessment,The Constitutional Court considered that the Federal Administrative Court du...
3,0,74,3968_chunk000,3,5.420974,Asylum Definition And Assessment,The case concerned an applicant from Pakistan who applied unsuccessfully for...
4,0,151,1648_chunk000,4,5.272908,Asylum Definition And Assessment,"The applicant applied for international protection on 13 January 2010, claim..."
5,0,39,4036_chunk001,5,5.107758,Asylum Definition And Assessment,The Supreme Administrative Corut had to determine whether the administrative...
6,0,127,3231_chunk001,6,5.107758,Asylum Definition And Assessment,"Finally, it stated that if bone examinations contravene such records and ass..."
7,0,447,129_chunk000,7,4.990283,Asylum Definition And Assessment,"In the absence of legally binding criteria to assess risk of absconding, app..."
8,0,473,196_chunk001,8,4.569867,Asylum Definition And Assessment,"Based on Chapter III of the Dublin III Regulation, the Office of the Refugee..."
9,0,3,5443_chunk001,9,4.475603,Asylum Definition And Assessment,It noted also that the lower court duly considered the admissibility of the ...


## 4. Inspect results for one query

In [4]:
sample_qid = results_df["qid"].iloc[0]
cols = ["qid", "query", "docno", "rank", "score", "text"]
results_df[results_df["qid"] == sample_qid][cols].head(10)

,qid,query,docno,rank,score,text
0,0,Asylum Definition And Assessment,151_chunk001,0,5.851850,None of them possessed documentary evidence supporting their claimed ages an...
1,0,Asylum Definition And Assessment,201_chunk001,1,5.638718,It was submitted on behalf of the applicant that a family could be a social ...
2,0,Asylum Definition And Assessment,5442_chunk001,2,5.495891,The Constitutional Court considered that the Federal Administrative Court du...
3,0,Asylum Definition And Assessment,3968_chunk000,3,5.420974,The case concerned an applicant from Pakistan who applied unsuccessfully for...
4,0,Asylum Definition And Assessment,1648_chunk000,4,5.272908,"The applicant applied for international protection on 13 January 2010, claim..."
5,0,Asylum Definition And Assessment,4036_chunk001,5,5.107758,The Supreme Administrative Corut had to determine whether the administrative...
6,0,Asylum Definition And Assessment,3231_chunk001,6,5.107758,"Finally, it stated that if bone examinations contravene such records and ass..."
7,0,Asylum Definition And Assessment,129_chunk000,7,4.990283,"In the absence of legally binding criteria to assess risk of absconding, app..."
8,0,Asylum Definition And Assessment,196_chunk001,8,4.569867,"Based on Chapter III of the Dublin III Regulation, the Office of the Refugee..."
9,0,Asylum Definition And Assessment,5443_chunk001,9,4.475603,It noted also that the lower court duly considered the admissibility of the ...


## 5. Optional: rerank the same run with monoT5

Reruns retrieval with `rerank_monot5=True` (reuses the already-built index,
since it lives at the same `index_dir` for this `docs_csv`/`retriever` pair).
Skip this cell if you don't need reranking.

In [5]:
rerank_config = RetrievalConfig(
    docs_csv=DOCS_CSV,
    topics_csv=TOPICS_CSV,
    retriever=RETRIEVER,
    top_k=TOP_K,
    language=LANGUAGE,
    rerank_monot5=True,
)
reranked_df = PyTerrierRetrievalPipeline(rerank_config).run()

reranked_output_path = rerank_config.output_csv or PyTerrierRetrievalPipeline(rerank_config)._default_output_path()
print(f"Reranked {len(reranked_df)} rows -> {reranked_output_path}")
reranked_df[reranked_df["qid"] == sample_qid][["qid", "query", "docno", "rank", "score", "text"]].head(10)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


monoT5: 100%|██████████| 72/72 [00:06<00:00, 10.63batches/s]

Reranked 285 rows -> euaa_asylum_report_queries_topic_names_qid_query_bm25_monot5.csv


,qid,query,docno,rank,score,text
0,0,Asylum Definition And Assessment,923_chunk000,0,-5.079955,An applicant must submit a request for protection with which he asserts targ...
1,0,Asylum Definition And Assessment,1648_chunk000,1,-5.786448,"The applicant applied for international protection on 13 January 2010, claim..."
2,0,Asylum Definition And Assessment,643_chunk000,2,-8.176759,As per the published press release: The Council of State validates the crite...
3,0,Asylum Definition And Assessment,3968_chunk000,3,-8.578689,The case concerned an applicant from Pakistan who applied unsuccessfully for...
4,0,Asylum Definition And Assessment,1086_chunk001,4,-8.697836,"Specifically, the assessment consisted of an initial observation by two offi..."
5,0,Asylum Definition And Assessment,563_chunk000,5,-9.154701,"The case concerns the moment at which an asylum application is lodged, in pa..."
6,0,Asylum Definition And Assessment,577_chunk000,6,-9.534427,This case concerns the validity of asylum application in accordance with § 2...
7,0,Asylum Definition And Assessment,921_chunk001,7,-9.767823,"More analytically, a female asylum seeker had upon arrival to Norway..."
8,0,Asylum Definition And Assessment,3277_chunk001,8,-9.789707,"When the asylum procedure starts, applicants are usually assigned a lawyer t..."
9,0,Asylum Definition And Assessment,108_chunk001,9,-10.124617,The applicant subsequently lodged a request for his asylum application to be...
